# 01 · Fine-tuning GPT-2 · enseñarle SQL

> El viaje completo del fine-tuning en un solo lab: coges un modelo que NO sabe una tarea,
> le haces full fine-tuning para que la aprenda, lo evaluas... y descubres lo que ha perdido por el camino.

## Qué aprendemos

1. **Full fine-tuning** : entrenar TODOS los pesos de un modelo (la forma original, antes de LoRA).
2. **Evaluar text-to-SQL** en tres niveles: a ojo, match exacto, y que el SQL ejecute de verdad.
3. **Olvido catastrofico** : ver, en vivo, como el modelo gana SQL pero pierde lo que sabia antes.

## Por qué importa

Es el primer eslabon de la cascada (Full FT -> SFT -> LoRA -> QLoRA). Hacerlo a pelo una vez te deja
sentir dos cosas que justifican todo lo que viene después: entrenar el modelo entero es **caro** y
**destruye** sus capacidades generales. El Lab 2 (mismo SQL, pero con LoRA) sera la cura.

## Conecta con

- **Slides de Fine-tuning · Deck 1** : "Cuando SI fine-tunear" (NL->SQL es el primer ejemplo) y "olvido catastrofico".
- **Slides de Fine-tuning · Deck 2** : SFT y el cascade.
- **Lab 02** : la misma tarea con LoRA, para comparar A/B.

## Stack

`gpt2` (124M) · `transformers` · `datasets` · dataset `b-mc2/sql-create-context`.

## Hardware

T4 (Colab Free) vale de sobra. GPT-2 es pequeño. Ejecución de referencia (oct-2026, T4): ~9 min en total, ~4 de entrenamiento.


---

## Setup


In [1]:
!pip install -qU transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 41.5/50.1 MB 245.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 264.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


In [2]:
import math, re, sqlite3, torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


---

# 1 · Cargamos GPT-2 y vemos lo malo que es en SQL

GPT-2 (2019) es un modelo **base**: solo sabe predecir el siguiente token. No ha visto pares
pregunta -> SQL y no sigue instrucciones. Vamos a comprobarlo.


In [3]:
MODEL = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token  # GPT-2 no trae pad token
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
print(f"{MODEL} cargado · {model.num_parameters()/1e6:.0f}M parametros")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

gpt2 cargado · 124M parametros


In [4]:
def build_prompt(question, context):
    return (
        "Given the database schema, write the SQL query.\n"
        f"Schema: {context}\n"
        f"Question: {question}\n"
        "SQL:"
    )

@torch.no_grad()
def generate(prompt, max_new_tokens=48):
    model.eval()  # dropout off · salidas reproducibles
    enc = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.split("\n")[0].strip()  # primera linea = la query

In [5]:
# Tres preguntas de SQL al GPT-2 base
demo = [
    ("How many employees are in the sales department?",
     "CREATE TABLE employees (name VARCHAR, salary INT, department VARCHAR)"),
    ("What is the average salary?",
     "CREATE TABLE employees (name VARCHAR, salary INT, department VARCHAR)"),
    ("List the names of products that cost more than 100.",
     "CREATE TABLE products (name VARCHAR, price INT)"),
]
print("=== GPT-2 BASE intentando SQL ===\n")
for q, ctx in demo:
    print("Q:", q)
    print("SQL generado:", generate(build_prompt(q, ctx)))
    print()

=== GPT-2 BASE intentando SQL ===

Q: How many employees are in the sales department?


SQL generado: SELECT * FROM employees WHERE name = 'Sales Manager'

Q: What is the average salary?


SQL generado: $1,000,000

Q: List the names of products that cost more than 100.


SQL generado: SELECT product_id, price FROM products WHERE name = 'C' AND price = '100' AND name = 'C' AND price = '100' AND name = 'C' AND price = '100' AND name = '



Como esperabamos: divaga, repite o inventa. **No sabe SQL.**

Pero ojo a lo que GPT-2 SI sabe hacer (texto general). Guarda estos resultados en la cabeza,
volveremos a ellos al final.


In [6]:
GENERAL_PROMPTS = [
    "Once upon a time, in a small village,",
    "The capital of France is",
    "To make a good cup of coffee, you need",
]

@torch.no_grad()
def general_completion(prompt, max_new_tokens=30):
    model.eval()
    enc = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

print("=== GPT-2 BASE en tareas generales ===\n")
for p in GENERAL_PROMPTS:
    print(general_completion(p), "\n")

=== GPT-2 BASE en tareas generales ===



Once upon a time, in a small village, a young man named Tzal, who had been a member of the village's council, was killed by a group of bandits. He was taken 



The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.

The French Republic is the capital 



To make a good cup of coffee, you need to make sure you have a good cup of coffee.

The best way to make a good cup of coffee is to make a cup of coffee 



Coherente (dentro de lo que es un modelo de 2019). **Esto es lo que sabe hacer hoy.**


---

# 2 · El dataset · text-to-SQL

`b-mc2/sql-create-context`: cada ejemplo trae la pregunta, el esquema (CREATE TABLE) y el SQL correcto.


In [7]:
ds = load_dataset("b-mc2/sql-create-context", split="train")
print(ds)
print("\nEjemplo:")
for k, v in ds[0].items():
    print(f"  {k}: {v}")

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})

Ejemplo:
  answer: SELECT COUNT(*) FROM head WHERE age > 56
  question: How many heads of the departments are older than 56 ?
  context: CREATE TABLE head (age INTEGER)


In [8]:
# Formato de entrenamiento: prompt + respuesta + salto de linea + fin de secuencia
# Ojo: GPT-2 no tiene token de padding y usamos pad = eos. El collator pone label -100 en todo
# token igual al pad, asi que el EOS final NO se entrena y el modelo nunca aprenderia a parar
# (repetiria "AND ... AND ..." hasta agotar tokens). El "\n" tras la query SI se entrena:
# le enseña a cortar, y generate() se queda con la primera linea.
def to_text(ex):
    return {"text": build_prompt(ex["question"], ex["context"]) + " " + ex["answer"] + "\n" + tok.eos_token}

ds = ds.map(to_text)

# Subconjuntos para que entrene rapido en T4
ds = ds.shuffle(seed=3407)
train_ds = ds.select(range(8000))
eval_ds  = ds.select(range(8000, 8300))   # 300 para evaluar
print("train:", len(train_ds), "· eval:", len(eval_ds))

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

train: 8000 · eval: 300


In [9]:
def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
collator = DataCollatorForLanguageModeling(tok, mlm=False)  # crea labels (LM causal)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

---

# 3 · Medimos el "antes" (GPT-2 base)

Antes de entrenar capturamos dos números que compararemos al final:
- **Accuracy de SQL** del modelo base (sera ~0%).
- **Perplejidad** del modelo en texto general (cuanto mas baja, mejor entiende el texto normal).


In [10]:
def normalize_sql(s):
    s = s.strip().rstrip(";").lower()
    return re.sub(r"\s+", " ", s)

@torch.no_grad()
def sql_accuracy(n=120, show=5):
    ok = 0
    for i, ex in enumerate(eval_ds.select(range(n))):
        pred = generate(build_prompt(ex["question"], ex["context"]))
        match = normalize_sql(pred) == normalize_sql(ex["answer"])
        ok += int(match)
        if i < show:   # vemos en directo los SQL generados
            print(f"[{'OK ' if match else 'FALLO'}] {ex['question']}")
            print(f"   generado: {pred}")
            print(f"   esperado: {ex['answer']}\n")
    return ok / n

GENERAL_TEXT = (
    "The history of artificial intelligence began in antiquity, with myths and stories "
    "of artificial beings endowed with intelligence by master craftsmen."
)

@torch.no_grad()
def perplexity(text):
    model.eval()
    enc = tok(text, return_tensors="pt").to(device)
    loss = model(**enc, labels=enc["input_ids"]).loss
    return math.exp(loss.item())

acc_base = sql_accuracy()
ppl_base = perplexity(GENERAL_TEXT)
print(f"SQL accuracy (base): {acc_base:.1%}")
print(f"Perplejidad en texto general (base): {ppl_base:.1f}")

[FALLO] What is the result of the game when the attendance is 62,078?
   generado: SELECT * FROM table_name_6 WHERE attendance = 62,078
   esperado: SELECT result FROM table_name_6 WHERE attendance = "62,078"



[FALLO] Which season had a home 2-3?
   generado: SELECT season FROM season WHERE season.season_id = season.season_id;
   esperado: SELECT season FROM table_name_2 WHERE home = "2-3"



[FALLO] On what Date is the match played on Clay Surface with a Score in the final of 6–2, 7–5?
   generado: SELECT * FROM table_name_32 WHERE score_in_the_final = 6;
   esperado: SELECT date FROM table_name_32 WHERE surface = "clay" AND score_in_the_final = "6–2, 7–5"



[FALLO] When Jim Thorpe of United States has a score of 71, what is the place?
   generado: The place is the name of the player.
   esperado: SELECT place FROM table_name_2 WHERE score = 71 AND country = "united states" AND player = "jim thorpe"



[FALLO] What is the custody level of the facility in Shelton?
   generado: The custody level is the level of the facility that is in the custody of the tenant.
   esperado: SELECT custody_level_s_ FROM table_25346763_1 WHERE location = "Shelton"



[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


SQL accuracy (base): 0.0%
Perplejidad en texto general (base): 44.3


---

# 4 · Full fine-tuning · entrenamos TODOS los pesos

GPT-2 es pequeño, así que podemos permitirnos el lujo del full fine-tuning: actualizamos
**los 124M de parámetros**. (En modelos grandes esto es inviable; por eso existe LoRA, Lab 2.)


In [11]:
args = TrainingArguments(
    output_dir="gpt2-sql",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    fp16=(device == "cuda"),
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=collator)
trainer.train()

Step,Training Loss
50,1.737331
100,1.232069
150,1.192707
200,1.149119
250,1.078124
300,1.081598
350,1.044162
400,1.033449
450,1.051482
500,1.034309


TrainOutput(global_step=2000, training_loss=0.9642219142913818, metrics={'train_runtime': 248.364, 'train_samples_per_second': 64.422, 'train_steps_per_second': 8.053, 'total_flos': 943235656704000.0, 'train_loss': 0.9642219142913818, 'epoch': 2.0})

---

# 5 · ¿Ahora sabe SQL?

Las MISMAS preguntas del paso 1, ahora con el modelo fine-tuneado.


In [12]:
print("=== GPT-2 FINE-TUNEADO en SQL ===\n")
for q, ctx in demo:
    print("Q:", q)
    print("SQL generado:", generate(build_prompt(q, ctx)))
    print()

=== GPT-2 FINE-TUNEADO en SQL ===

Q: How many employees are in the sales department?


SQL generado: SELECT COUNT(*) FROM employees WHERE salary > 0 AND department = "Sales"

Q: What is the average salary?


SQL generado: SELECT AVG(name) FROM employees WHERE salary > 0

Q: List the names of products that cost more than 100.


SQL generado: SELECT name FROM products ORDER BY price DESC LIMIT 100



---

# 6 · Evaluación · en tres niveles

El "a ojo" del paso 5 ya impresiona, pero no es medir. Subimos el rigor:

1. **Match normalizado** : ¿el SQL generado es igual al de referencia? (rápido, pero estricto)
2. **Validez de ejecución** : ¿el SQL ejecuta contra el esquema sin error? (caza columnas inventadas)
3. **Execution accuracy** : ¿devuelve el mismo resultado que el SQL correcto? (la métrica de verdad)


In [13]:
# Nivel 1 · match EXACTO (muy estricto). Arriba veras los SQL generados.
acc_ft = sql_accuracy()   # imprime ejemplos: fijate que son BUENOS aunque no identicos
print(f"\nMatch exacto · base: {acc_base:.1%}  ->  fine-tuneado: {acc_ft:.1%}")
print("Ojo: el match exacto castiga SQL validos pero escritos distinto (otro alias, otro orden).")
print("Mira los SQL de arriba: son buenos. Por eso medimos tambien si EJECUTAN (Nivel 2)")
print("y si dan el MISMO resultado (Nivel 3), que es lo que de verdad importa.")

[OK ] What is the result of the game when the attendance is 62,078?
   generado: SELECT result FROM table_name_6 WHERE attendance = "62,078"
   esperado: SELECT result FROM table_name_6 WHERE attendance = "62,078"



[OK ] Which season had a home 2-3?
   generado: SELECT season FROM table_name_2 WHERE home = "2-3"
   esperado: SELECT season FROM table_name_2 WHERE home = "2-3"



[OK ] On what Date is the match played on Clay Surface with a Score in the final of 6–2, 7–5?
   generado: SELECT date FROM table_name_32 WHERE surface = "clay" AND score_in_the_final = "6–2, 7–5"
   esperado: SELECT date FROM table_name_32 WHERE surface = "clay" AND score_in_the_final = "6–2, 7–5"



[OK ] When Jim Thorpe of United States has a score of 71, what is the place?
   generado: SELECT place FROM table_name_2 WHERE score = 71 AND country = "united states" AND player = "jim thorpe"
   esperado: SELECT place FROM table_name_2 WHERE score = 71 AND country = "united states" AND player = "jim thorpe"



[OK ] What is the custody level of the facility in Shelton?
   generado: SELECT CUSTODY_level_s_ FROM table_25346763_1 WHERE location = "Shelton"
   esperado: SELECT custody_level_s_ FROM table_25346763_1 WHERE location = "Shelton"




Match exacto · base: 0.0%  ->  fine-tuneado: 62.5%
Ojo: el match exacto castiga SQL validos pero escritos distinto (otro alias, otro orden).
Mira los SQL de arriba: son buenos. Por eso medimos tambien si EJECUTAN (Nivel 2)
y si dan el MISMO resultado (Nivel 3), que es lo que de verdad importa.


In [14]:
# Nivel 2 · validez de ejecucion contra el esquema (tabla vacia)
def executes_ok(sql, create_stmt):
    try:
        con = sqlite3.connect(":memory:")
        con.executescript(create_stmt)         # crea el esquema
        con.execute(sql)                        # intenta la query
        con.close()
        return True
    except Exception:
        return False

valid = 0
n = 100
for ex in eval_ds.select(range(n)):
    pred = generate(build_prompt(ex["question"], ex["context"]))
    if executes_ok(pred, ex["context"]):
        valid += 1
print(f"SQL que ejecuta sin error contra el esquema: {valid/n:.1%}")

SQL que ejecuta sin error contra el esquema: 91.0%


In [15]:
# Nivel 3 · execution accuracy en un ejemplo con datos sinteticos
# (sql-create-context no trae filas; las inventamos para un caso concreto)
create = "CREATE TABLE employees (name TEXT, salary INT, department TEXT)"
rows = [("Ana", 50000, "sales"), ("Luis", 60000, "sales"), ("Eva", 70000, "engineering")]

con = sqlite3.connect(":memory:")
con.executescript(create)
con.executemany("INSERT INTO employees VALUES (?,?,?)", rows)

gold = "SELECT COUNT(*) FROM employees WHERE department = 'sales'"
pred = generate(build_prompt("How many employees are in the sales department?", create))
print("SQL generado:", pred)

def run(sql):
    try:
        return con.execute(sql).fetchall()
    except Exception as e:
        return f"ERROR: {e}"

print("resultado gold:", run(gold))
print("resultado pred:", run(pred))
print("coinciden:", run(gold) == run(pred))
con.close()

SQL generado: SELECT COUNT(*) FROM employees GROUP BY name
resultado gold: [(2,)]
resultado pred: [(1,), (1,), (1,)]
coinciden: False


Esos tres niveles son justo el espectro de rigor que veras en el **módulo de Evaluación**:
desde mirar a ojo hasta ejecutar de verdad.


---

# 7 · El precio: olvido catastrofico

Ganamos SQL. ¿A costa de que? Volvemos a los prompts generales del paso 1, ahora con el modelo
fine-tuneado.


In [16]:
print("=== El modelo fine-tuneado en tareas generales ===\n")
for p in GENERAL_PROMPTS:
    print(general_completion(p), "\n")

=== El modelo fine-tuneado en tareas generales ===



Once upon a time, in a small village, what is the time?
SQL: SELECT time FROM table_name_92 WHERE time = "time"
Question: What is the time of 



The capital of France is the capital of the country's largest oil company.

The country's largest oil company is Total, which has a total of more than 1, 



To make a good cup of coffee, you need to make the cup.
Schedule the cup.
Schedule the cup.
Question: What is the cup size?
SQL: SELECT 



Donde antes había frases coherentes, ahora aparece SQL o ruido. **El modelo ha olvidado lo que
sabia.** Lo cuantificamos con la perplejidad en texto general (mas alta = peor):


In [17]:
ppl_ft = perplexity(GENERAL_TEXT)
print(f"Perplejidad en texto general · base: {ppl_base:.1f}  ->  fine-tuneado: {ppl_ft:.1f}")
print(f"\nResumen del viaje:")
print(f"  SQL accuracy : {acc_base:.1%}  ->  {acc_ft:.1%}   (gano la tarea)")
print(f"  Perplejidad  : {ppl_base:.1f}  ->  {ppl_ft:.1f}   (perdio lo general)")

Perplejidad en texto general · base: 44.3  ->  fine-tuneado: 433.5

Resumen del viaje:
  SQL accuracy : 0.0%  ->  62.5%   (gano la tarea)
  Perplejidad  : 44.3  ->  433.5   (perdio lo general)


---

# Cierre · lo que nos llevamos

- Hicimos **full fine-tuning** de GPT-2 y le enseñamos SQL de la nada.
- Lo evaluamos en serio: match, validez de ejecución y execution accuracy.
- Y vimos el coste: **olvido catastrofico** (gano SQL, perdio lo general) + entrenamos los 124M de pesos.

## Hacia el Lab 2

¿Y si pudieramos enseñarle SQL **sin romper** lo que ya sabia, y entrenando una fracción minuscula
de los pesos? Eso es **LoRA**. En el Lab 2 hacemos la MISMA tarea con la MISMA evaluación, y comparamos:
una accuracy de SQL cercana (en la ejecución de referencia: ~62% aquí con full FT, ~51% con LoRA) entrenando menos del 1% de los pesos, y el modelo base queda intacto. Esa es la cura.
